In [2]:
# membuat model dari program customer churn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

file_path = 'train.csv'
df = pd.read_csv(file_path)
print("berhasil membaca file directory")
df.head()

berhasil membaca file directory


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [3]:
from sklearn.preprocessing import LabelEncoder
# melakukan pemisahan data
X = df.drop(columns=["Churn"])
# gunakan label encoder
le = LabelEncoder()
new_y = le.fit_transform(df["Churn"])
y = new_y
print(X.head())

   id  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0   0    Male              0     Yes        Yes      29          Yes   
1   1    Male              0     Yes        Yes      58          Yes   
2   2    Male              0     Yes         No      58          Yes   
3   3  Female              0      No         No       1          Yes   
4   4  Female              0      No         No       1          Yes   

  MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection  \
0            No             DSL            Yes           No              Yes   
1            No             DSL            Yes          Yes               No   
2           Yes     Fiber optic             No          Yes               No   
3            No     Fiber optic             No           No               No   
4            No     Fiber optic             No           No               No   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0       

In [4]:
# lakukan pemisahan untuk data train dan test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
107052,107052,Female,0,Yes,Yes,67,Yes,Yes,DSL,Yes,Yes,Yes,Yes,No,Yes,Two year,No,Mailed check,83.00,5293.4
249213,249213,Female,0,No,No,47,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),19.50,929.3
206374,206374,Male,0,Yes,No,65,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.20,6844.5
347184,347184,Female,0,Yes,Yes,51,Yes,No,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,101.15,5264.5
99430,99430,Female,0,Yes,Yes,59,Yes,No,Fiber optic,Yes,Yes,No,No,No,No,Month-to-month,No,Bank transfer (automatic),81.30,4859.5


## classification data
mengklasifikasikan data numerical dan data categorical

In [5]:
numerical_cols = list(X_train.select_dtypes(include=['int64', 'float64']).columns)
print(f"Numerical columns: {numerical_cols}")
# kolom categorical
categorical_cols = list(X_train.select_dtypes(include=['object']).columns)
print(f"Categorical columns: {categorical_cols}")
categorical_nunique = [X_train[col].nunique() for col in categorical_cols]
d = dict(zip(categorical_cols, categorical_nunique))
print(f"Number of unique values in each categorical column: {d}")
# banyak data kategori yang harus di visualisasikan

Numerical columns: ['id', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Number of unique values in each categorical column: {'gender': 2, 'Partner': 2, 'Dependents': 2, 'PhoneService': 2, 'MultipleLines': 3, 'InternetService': 3, 'OnlineSecurity': 3, 'OnlineBackup': 3, 'DeviceProtection': 3, 'TechSupport': 3, 'StreamingTV': 3, 'StreamingMovies': 3, 'Contract': 3, 'PaperlessBilling': 2, 'PaymentMethod': 4}


In [6]:
# membuat klasifikasi object
df_classification = X_train[categorical_cols]
df_classification.head()
# semuanya memakai one hot encoder

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod
107052,Female,Yes,Yes,Yes,Yes,DSL,Yes,Yes,Yes,Yes,No,Yes,Two year,No,Mailed check
249213,Female,No,No,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic)
206374,Male,Yes,No,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check
347184,Female,Yes,Yes,Yes,No,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check
99430,Female,Yes,Yes,Yes,No,Fiber optic,Yes,Yes,No,No,No,No,Month-to-month,No,Bank transfer (automatic)


In [7]:
# membuat pipeline untuk preprocessing data
from sklearn.compose import ColumnTransformer
# pipeline untuk data numerik
OH_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)
imputer_numerical = SimpleImputer(strategy='mean')
imputer_categorical = SimpleImputer(strategy='most_frequent')

onehot_transformer = Pipeline(steps=[
    ('imputer', imputer_categorical),
    ('encoder', OH_encoder)
])

# mulai melakukan preprocess dengan pipeline
preprocessing = ColumnTransformer(transformers=[
    ('numerical_cols', imputer_numerical, numerical_cols),
    ('categorical_cols', onehot_transformer, categorical_cols)
])

print("berhasil melakukan preprocessing")

berhasil melakukan preprocessing


In [8]:
# mulai memakai model dengan pipeline
# hitung rasio imbalance
negative_scale = (y_train == 0).sum()
positive_scale = (y_train == 1).sum()
ratio = negative_scale / positive_scale

model_xgb = XGBClassifier(
    n_estimators=200,
    scale_pos_weight=ratio,
    learning_rate=0.05,
)

my_pipeline = Pipeline(steps=[
    ('preprocess', preprocessing),
    ('model', model_xgb)
])

print("sukses membuat mesin pipeline")

sukses membuat mesin pipeline


In [9]:
# mulai melakukan proses terhadap modelnya
from sklearn.metrics import accuracy_score
my_pipeline.fit(X_train, y_train)

prediction = my_pipeline.predict(X_test)
score = accuracy_score(y_test, prediction)

print(prediction)
print(f"Score nya adalah : {score}")
# mendapatkan score di 81%

[0 0 0 ... 0 1 1]
Score nya adalah : 0.8193438181068504


In [10]:
# melanjutkan pushing dengan mengubah label encoder terlebih dahulu
prediction_text = le.inverse_transform(prediction)
print(prediction_text)
# membuat feature importance nya
import shap
# buat explainer
# explainer = shap.TreeExplainer(model_xgb)

['No' 'No' 'No' ... 'No' 'Yes' 'Yes']


/Users/syawqiarroyan/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# mencoba dengan data test
dft = pd.read_csv('test.csv')
# preprocess terlebih dahulu
prediction_test = my_pipeline.predict(dft)
print(prediction_test)
prediction_test_text = le.inverse_transform(prediction_test)
print(prediction_test)

[0 0 0 ... 1 0 1]
[0 0 0 ... 1 0 1]


In [13]:
# masukkan ke dalam file submission
submission = pd.DataFrame({
    'id' : dft['id'],
    'Churn' : prediction_test
})

submission.to_csv('submission.csv', index=False)
print("File submission berhasil dibuat")

File submission berhasil dibuat


In [1]:
# membuat perbaikan terhadap model push
print("ini adalah file commit baru")

ini adalah file commit baru


In [3]:
# melakukan feature engineering
# menambahkan kolom baru
df.head()


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
df['Charge_Group'] = pd.cut(
    df['MonthlyCharges'],
    bins=[0,30,70, df['MonthlyCharges'].max()],
    labels=['low', 'medium', 'high']
)

df['Charge_Group']

0         medium
1         medium
2           high
3         medium
4           high
           ...  
594189      high
594190      high
594191       low
594192      high
594193    medium
Name: Charge_Group, Length: 594194, dtype: category
Categories (3, object): ['low' < 'medium' < 'high']

In [7]:
df.head()
# lakukan encoding
df['Charge_Group_Encoded'] = df['Charge_Group'].map({
    'low': 0,
    'medium' : 1,
    'high' : 2
})

df = df.drop(columns='Charge_Group')

df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Charge_Group_Encoded
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No,1
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No,1
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No,2
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes,1
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes,2


In [8]:
# menambahkan tenure dan juga yang telah dibuat pada saat encoding
df['Total_Charges'] = df['tenure'] * df['MonthlyCharges']

df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Charge_Group_Encoded,Total_Charges
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,No,No,One year,Yes,Mailed check,60.10,1653.85,No,1,1742.90
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No,1,4031.00
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No,2,5823.20
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes,1,69.70
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes,2,70.45


In [11]:
# menambahkan kembali kolom yang diperlukan
df['Avg_Monthly_Paid'] = df['TotalCharges'] / df['tenure']
# melakukan cutting
df['Avg_Monthly_Paid_Group'] = pd.cut(
    df['Avg_Monthly_Paid'],
    bins=[0,30,90, df['Avg_Monthly_Paid'].max()],
    labels=['low_risk', 'medium_risk', 'high_risk']
)

df['Avg_Monthly_Group_Encoded'] = df['Avg_Monthly_Paid_Group'].map({
    'low_risk' : 0,
    'medium_risk' : 1,
    'high_risk' : 2
})

df = df.drop(columns='Avg_Monthly_Paid_Group')

df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Charge_Group_Encoded,Total_Charges,Avg_Monthly_Paid,Avg_Monthly_Group_Encoded
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,One year,Yes,Mailed check,60.10,1653.85,No,1,1742.90,57.029310,1
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,Two year,No,Credit card (automatic),69.50,3778.20,No,1,4031.00,65.141379,1
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,Month-to-month,Yes,Electronic check,100.40,5841.35,No,2,5823.20,100.712931,2
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,69.70,70.70,Yes,1,69.70,70.700000,1
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.45,70.45,Yes,2,70.45,70.450000,1


In [ ]:
# mulai melakukan preprocessing dengan pipeline